<div>

  <b>Escuela Politécnica Nacional</b><br>
  <small>Facultad de Ingeniería de Sistemas</small><br>
  <small>Ingeniería en Ciencias de la Computación</small>

  <hr>

  <div style="display:flex; justify-content:space-between;">
    <div>
      Estudiante: <b>Mateo Cumbal</b><br>
      Fecha: <b>2026-05-13</b>
    </div>
    <div style="text-align:right;">
      Asignatura: <b>Recuperación de la Información</b><br>
      Paralelo: <b>GR1CC</b>
    </div>
  </div>

  <hr>

  <div style="text-align:center;">
    <h1><b>Ejercicio 5 — Espacio Vectorial</b></h1>
  </div>

</div>

## Objetivo
Implementar un Sistema de Recuperación de Información completo sobre un corpus de Wikipedia, comparando los modelos TF-IDF y BM25 para un conjunto de 10 queries.

In [1]:
import sys
import string
import numpy as np
import pandas as pd
from collections import Counter

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append("../")
import libreria as ir

## Parte 0: Carga y Preprocesamiento del Corpus

Se utiliza el dataset _Wikipedia Text Corpus for NLP and LLM Projects_ de Kaggle (10,859 artículos).

In [2]:
# Carga del CSV
df = pd.read_csv("../wikipedia/wikipedia_text_corpus.csv")

# Separar título y cuerpo: primera línea es el título, el resto es el artículo
df[["titulo", "cuerpo"]] = df["text"].str.split("\n\n", n=1, expand=True)
df = df.drop(columns=["Unnamed: 0", "text"]).reset_index(drop=True)

print(f"Documentos cargados: {len(df)}")
df.head(3)

Documentos cargados: 10859


,titulo,cuerpo
0,Anovo,Anovo (formerly A Novo) is a computer services...
1,Battery indicator,A battery indicator (also known as a battery g...
2,Bob Pease,"Robert Allen Pease (August 22, 1940Â â€“ June ..."


In [3]:
# Preprocesamiento: lowercase, eliminación de puntuación, stopwords y stemming
stop_words = set(stopwords.words("english"))
puntuacion = set(string.punctuation)
stemmer    = PorterStemmer()

def preprocesar(texto):
    if not isinstance(texto, str):
        return []
    tokens = texto.lower().split()
    tokens = [t.strip(string.punctuation) for t in tokens]
    tokens = [t for t in tokens if t and t not in stop_words and t not in puntuacion]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

df["tokens"]       = df["cuerpo"].apply(preprocesar)
df["texto_limpio"] = df["tokens"].apply(lambda t: " ".join(t))

df[["titulo", "tokens"]].head(3)

,titulo,tokens
0,Anovo,"[anovo, formerli, novo, comput, servic, compan..."
1,Battery indicator,"[batteri, indic, also, known, batteri, gaug, d..."
2,Bob Pease,"[robert, allen, peas, august, 22, 1940â, â€“, ..."


## Parte 1: Recuperación con TF-IDF

Se construye la matriz TF-IDF usando sparse matrix para manejar el vocabulario de alrededor de 179k términos. La búsqueda se realiza por similitud coseno entre la query transformada y los documentos.

In [4]:
# Construcción de la matriz TF-IDF (sparse)
vectorizer   = TfidfVectorizer(tokenizer=lambda x: x.split(), token_pattern=None)
tfidf_matrix = vectorizer.fit_transform(df["texto_limpio"])

print(f"Dimensiones matriz TF-IDF: {tfidf_matrix.shape}")

Dimensiones matriz TF-IDF: (10859, 179280)


In [5]:
def buscar_tfidf(query, top_k=5):
    tokens_query = preprocesar(query)
    texto_query  = " ".join(tokens_query)
    query_vec    = vectorizer.transform([texto_query])

    similitudes = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(similitudes)[::-1][:top_k]

    return pd.DataFrame({
        "rank"  : range(1, top_k + 1),
        "titulo": df["titulo"][top_indices].values,
        "score" : similitudes[top_indices]
    })

In [7]:
queries = [
    "artificial intelligence machine learning",
    "world war history",
    "climate change environment",
    "music theory instruments",
    "space exploration planets",
    "ancient civilizations egypt",
    "medicine human body diseases",
    "economics financial markets",
    "literature poetry writers",
    "sports olympic games",
]

# Recuperación TF-IDF para las 10 queries
resultados_tfidf = {q: buscar_tfidf(q, top_k=5) for q in queries}

# Ejemplo
resultados_tfidf["artificial intelligence machine learning"]

,rank,titulo,score
0,1,Outline of machine learning,0.623181
1,2,Artificial psychology,0.452735
2,3,Digital learning,0.365550
3,4,Teaching machine,0.364160
4,5,Accounting intelligence,0.340649


## Parte 2: Recuperación con BM25

A diferencia de TF-IDF, penaliza documentos inusualmente largos.

In [9]:
# Preparación de estructuras BM25
docs_tokens = df["tokens"].tolist()
N_docs      = len(docs_tokens)
doc_lengths = [len(doc) for doc in docs_tokens]
avgdl       = np.mean(doc_lengths)

# TF crudo por documento
tf_bm25 = [Counter(doc) for doc in docs_tokens]

# Document frequency y IDF BM25
df_freq = Counter()
for tf_doc in tf_bm25:
    df_freq.update(tf_doc.keys())

idf_bm25 = {
    term: np.log((N_docs - freq + 0.5) / (freq + 0.5) + 1)
    for term, freq in df_freq.items()
}

# Hiperparámetros
k1 = 1.2
b  = 0.75

In [10]:
def bm25_score(tf_doc, doc_length, query_terms):
    """Calcula el score BM25 de un documento dado un query."""
    score = 0.0
    for term in query_terms:
        if term not in idf_bm25:
            continue
        tf_term = tf_doc.get(term, 0)
        score  += idf_bm25[term] * (tf_term * (k1 + 1)) / (
                    tf_term + k1 * (1 - b + b * (doc_length / avgdl))
                  )
    return score

In [11]:
def buscar_bm25(query, top_k=5):
    """Recupera los top_k documentos más relevantes usando BM25."""
    query_terms = preprocesar(query)
    scores = np.array(
        [bm25_score(tf_bm25[i], doc_lengths[i], query_terms) for i in range(N_docs)]
    )
    top_indices = np.argsort(scores)[::-1][:top_k]

    return pd.DataFrame(
        {
            "rank": range(1, top_k + 1),
            "titulo": df["titulo"][top_indices].values,
            "score": scores[top_indices],
        }
    )

In [12]:
# Recuperación BM25 para las 10 queries
resultados_bm25 = {q: buscar_bm25(q, top_k=5) for q in queries}

# Ejemplo
resultados_bm25["artificial intelligence machine learning"]

,rank,titulo,score
0,1,Outline of machine learning,19.945665
1,2,Trace3,18.268175
2,3,Insilico Medicine,18.017004
3,4,Cybernetics and Systems,17.949456
4,5,DoceboLMS,17.902956


## Parte 3: Comparación de resultados

Se presentan los rankings de ambos modelos para las 10 queries, permitiendo observar coincidencias y divergencias en los documentos recuperados.

In [13]:
def comparar_tablas(query):
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    print("\nTF-IDF:")
    display(resultados_tfidf[query])
    print("\nBM25:")
    display(resultados_bm25[query])

for q in queries:
    comparar_tablas(q)


Query: artificial intelligence machine learning

TF-IDF:


,rank,titulo,score
0,1,Outline of machine learning,0.623181
1,2,Artificial psychology,0.452735
2,3,Digital learning,0.365550
3,4,Teaching machine,0.364160
4,5,Accounting intelligence,0.340649



BM25:


,rank,titulo,score
0,1,Outline of machine learning,19.945665
1,2,Trace3,18.268175
2,3,Insilico Medicine,18.017004
3,4,Cybernetics and Systems,17.949456
4,5,DoceboLMS,17.902956



Query: world war history

TF-IDF:


...

,rank,titulo,score
0,1,Pauline Gower,13.042741
1,2,Printing in Tamil language,11.734736
2,3,Alice Perry,11.101060
3,4,Andy Rathbone,10.779801
4,5,Burma-Shave,10.571200



Query: sports olympic games

TF-IDF:


,rank,titulo,score
0,1,Digital Beijing Building,0.326459
1,2,Games and learning,0.321832
2,3,List of video games derived from mods,0.312170
3,4,Digital distribution in video games,0.311888
4,5,Technology doping,0.295573



BM25:


,rank,titulo,score
0,1,Technology doping,19.565634
1,2,R J Mitchell Wind Tunnel,14.852083
2,3,History of graphic design,14.649022
3,4,Valley Preferred Cycling Center,14.186101
4,5,Digital Beijing Building,13.952290


## Conclusiones

**Coincidencia en documentos recuperados:**
Las queries *climate change environment*, *ancient civilizations egypt* y *economics financial markets* presentan 3 o más documentos en común entre ambos modelos, aunque en diferente orden. En el resto de queries la coincidencia es de 0 o 1 documento.

**Diferencias en ranking:**
Cuando ambos modelos recuperan los mismos documentos, el orden difiere. Por ejemplo en *climate change*, TF-IDF coloca "US Climate Reference Network" en rank 1 mientras BM25 lo ubica en rank 3, poniendo primero "Minister of Energy, Science, Technology, Environment...".

**Diferencias en escala de scores:**
TF-IDF produce scores entre 0 y 1 (similitud coseno), mientras BM25 genera scores en rangos mayores (~10-20). Para comparar ambos modelos fue necesario normalizar los valores.

**Documentos exclusivos por modelo:**
En queries como *music theory instruments* y *artificial intelligence machine learning*, BM25 recupera documentos que TF-IDF no considera relevantes ("The Typewriter", "Trace3", "DoceboLMS"), y viceversa. Esto indica que cada modelo pondera la relevancia de forma distinta para los mismos términos.

**Scores bajos como indicador:**
En la query *literature poetry writers*, TF-IDF produce scores muy bajos (máximo 0.27, el resto por debajo de 0.12), lo que indica baja similitud entre la query y los documentos recuperados. BM25 en cambio genera scores similares a las demás queries (~10-13), sugiriendo que es menos sensible a esta situación.